In [ ]:
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
import pandas as pd 
from typing import Literal

In [ ]:
### Test relative gain based flat dev on a post dark signal subtraction L0 observation 

In [ ]:
def make_relative_gain_flat(
        obs_image: np.ndarray,
        direction: Literal["left", "right"],
        left_cutoff: int,
        right_cutoff: int,
        readout_cols: list,
        saturation=None,
        min_valid_frac=0.5,
        min_dynamic_range=None
) -> np.ndarray:
    """
    Use the relative difference between adjacent columns to make a flat field
    normalized to the gain (mult factor) between columns being 1 per channel.
    Args:
        obs_image: Full observation, could be in DN or radiance. Must be dark
            signal subtracted otherwise the results will be dominated by dark
            signal.
        direction: Which adjacent column to use for relative diff (left or
            right). Defaults to left.
        left_cutoff: Left side of detector to mask (not illuminated).
        right_cutoff: Right side of detector to mask (not illuminated).
        readout_cols: Readout cols to mask.
        saturation: Mask high values above this threshold.
        min_valid_frac: What fraction of lines should be valid (e.g. not
            saturated or 0 std) to use in the flat?
        min_dynamic_range: We should ignore columns with variance below this.
    """
    obs_image[:, :, 80] = obs_image[:, :, 81]
    obs_image[:, :, 160] = obs_image[:, :, 161]
    obs_image[:, :, 240] = obs_image[:, :, 241]
    
    obs_image = obs_image[:, :, 9:313]

    nlines, nbands, ncols = obs_image.shape
    gain = np.ones((nbands, ncols), dtype=np.float64)
    
    for b in range(nbands):
        band_data = obs_image[:, b, :]  # copy per band for masking
        # mask bad pixels
        bad = ~np.isfinite(band_data) | (band_data <= 0)
        # mask "saturated" pixels
        # if saturation is not None:
        #     bad |= (band_data >= saturation)
        # # mask bad cols, cutoff dark signal / scattered light edges
        # bad[:, :left_cutoff] = True
        # bad[:, right_cutoff:] = True
        # # if readout_cols:
        # #     bad[:, readout_cols] = True
        # band_data = np.where(bad, np.nan, band_data)

        # we put the DN in log space bc log x - log y = log(x/y)
        log_data = np.log(band_data)
        if direction == "right":
            log_ratio_pairs = log_data[:, :-1] - log_data[:, 1:]
        else:
            log_ratio_pairs = log_data[:, 1:] - log_data[:, :-1]

        # don't use columns with low std? we don't want noise to be
        # interpreted as real, reoccurring column to column differences
        if min_dynamic_range is not None:
            local_std = np.nanstd(log_data, axis=1, keepdims=True)
            flat_rows = local_std < min_dynamic_range
            log_ratio_pairs = np.where(flat_rows, np.nan, log_ratio_pairs)

        # Median across lines, ignoring NaNs
        # valid_frac = np.nanmean(np.isfinite(log_ratio_pairs), axis=0)
        with np.errstate(invalid="ignore"):
            median_log_ratio = np.nanmedian(log_ratio_pairs, axis=0)

        # use 0.0 at bad pixels / where there are not enough downline samples
        # to get a good estimate of relative gain
        # median_log_ratio = np.where(valid_frac >= min_valid_frac,
        #                             median_log_ratio, 0.0)
        # median_log_ratio = np.nan_to_num(median_log_ratio, nan=0.0)

        if direction == "right":
            combined_gains = np.concatenate((
                -np.cumsum(median_log_ratio[::-1])[::-1],
                [0.0]
            ))
        else:
            combined_gains = np.concatenate((
                [0.0],
                np.cumsum(median_log_ratio)
            ))

        # normalize to median gain of 1
        combined_gains -= np.median(combined_gains)
        gain[b, :] = np.exp(combined_gains)
    return gain


def get_relative_gain_flat(obs_image: np.ndarray, left_col_cutoff, right_col_cutoff, read_out_cols) -> np.ndarray:
    """
    Take the average of two flats based on the relative gain to the left and
    to the right of each column.

    Kind of like histogram matching but not using the actual histogram of each
    column, instead using each column's relationship to its neighbors?

    This will not work if there are long, vertical, geographic / geologic
    features on the Moon that are the width of a M3 column and extend for most
    of an observation.
    """

    flat_1 = make_relative_gain_flat(
        obs_image,
        direction='left',
        left_cutoff=left_col_cutoff,
        right_cutoff=right_col_cutoff,
        readout_cols=read_out_cols,
        min_valid_frac=0.5,
    )

    flat_2 = make_relative_gain_flat(
        obs_image,
        direction='right',
        left_cutoff=left_col_cutoff,
        right_cutoff=right_col_cutoff,
        readout_cols=read_out_cols,
        min_valid_frac=0.5,
    )

    return ((1.0 / flat_1) + flat_2) / 2.0


In [ ]:
obs_id = "m3g20090210t012132"
left_col_cutoff = 9 
right_col_cutoff = 313
read_out_cols = [0, 80, 160, 240] 

metadata = pd.read_csv("/home/bekah/m3-pipeline-dev/obs_to_cal_file_mapping/obs_cal_info.csv") 
metadata = metadata[metadata['obs_id'] == obs_id.upper()]
if len(metadata) == 0:
    print("This is not a valid observation ID, although it could be a dark"
              " signal observation.")
if len(metadata) > 1:
    metadata = metadata.loc[metadata['version'].str.extract(r'(\d+)')[0].astype(int).idxmax()]

# look up dark signal observation 
dark_id = metadata['dark_signal_id'].lower()

# look up OG flat field ID 
flat_id = metadata['flat_field_id'].lower()

print(f"Dark ID: {dark_id}")
print(f"Flat ID: {flat_id}")

In [ ]:
# load dark signal observation 
dark_path = f"/home/bekah/m3-pipeline-dev/data/dark/darks_global/{dark_id}_l0.fits"
with fits.open(dark_path) as hdul:
    dark = hdul[0].data.transpose(1,0,2)[5:-5, :, :] # crop beginning and end 
dark = np.mean(dark, axis=0) 

# load observation 
obs_path = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{obs_id}_l0.fits"
with fits.open(obs_path) as hdul:
    obs_image = hdul[0].data.transpose(1,0,2)

# subtract dark signal 
obs_image = obs_image - dark[np.newaxis, :, :] 

In [ ]:
# make their flat field 

# load lab flat 
lab_path = "/home/bekah/m3-pipeline-dev/cal_files/lab_flat_field_global.fits"
with fits.open(lab_path) as hdul:
    lab_flat = hdul[0].data
    
# load their flat 
their_flat_path = f"/home/bekah/m3-pipeline-dev/data/flats/{flat_id}_ff.fits"
with fits.open(their_flat_path) as hdul:
    header = hdul[0].header.copy()
    their_flat = hdul[0].data
    
# multiply & save 
combo_flat = lab_flat / their_flat 
fits.PrimaryHDU(data=combo_flat, header=header).writeto(f"new_flat_comp/{obs_id}_combo_og_flat.fits", overwrite=True) 

In [ ]:
# make new flat 

new_flat = get_relative_gain_flat(obs_image, left_col_cutoff, right_col_cutoff, read_out_cols)

fits.PrimaryHDU(data=new_flat, header=header).writeto(f"new_flat_comp/{obs_id}_new_flat.fits", overwrite=True) 

In [ ]:
# apply flat 

obs_image = obs_image[:, :, 9:313] * new_flat[np.newaxis, :, :]

fits.PrimaryHDU(data=obs_image, header=header).writeto(f"new_flat_comp/{obs_id}_flatted_det_pov.fits", overwrite=True) 
fits.PrimaryHDU(data=obs_image.transpose(1,0,2)).writeto(f"new_flat_comp/{obs_id}_flatted_norm_pov.fits", overwrite=True) 

In [ ]:
# compare flats 

fits.PrimaryHDU(data=new_flat/combo_flat[:, 9:313], header=header).writeto(f"new_flat_comp/{obs_id}_myflat_div_theirflat.fits", overwrite=True) 
fits.PrimaryHDU(data=new_flat-combo_flat[:, 9:313], header=header).writeto(f"new_flat_comp/{obs_id}_myflat_minus_theirflat.fits", overwrite=True) 